# VisuNav: Familiar-Person Recognition
**Assistive Navigation for the Visually Impaired**

This module provides offline, real-time facial recognition to identify consenting familiar contacts (e.g., friends, family, caregivers) and reject unknown individuals.

### Architecture Pipeline:
1. **Face Detection & Alignment:** Multi-task Cascaded Convolutional Networks (`MTCNN`).
2. **Feature Extraction:** Pre-trained `InceptionResnetV1` (VGGFace2) producing normalized 512-dimensional embeddings.
3. **Contact Matching:** Cosine similarity with strict confidence thresholding ($>0.75$) and margin separation ($>0.08$) to prevent false positives.
4. **Assisted Output:** Spatial audio announcements (`pyttsx3`) indicating the recognized person and their relative direction (left, center, right).
5. **Visual Verification:** Inline `matplotlib` inference on test images with annotated bounding boxes.

In [ ]:
# Environment setup & Hyperparameter configuration
import json, time, queue, threading
from pathlib import Path
from collections import deque
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from facenet_pytorch import MTCNN, InceptionResnetV1

# Hardware acceleration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[VisuNav] Using hardware device: {DEVICE}")

# File paths
DB_PATH = Path("private_face_db/contacts.json")
ENROLLMENT_DIR = Path("enrollment")
TEST_FACES_DIR = Path("test_faces")

# Recognition & Quality Thresholds (calibrated for mobile / compressed photos)
THRESHOLD = 0.75            # Cosine similarity threshold for confident identification
MARGIN = 0.08               # Minimum gap required between best match and runner-up
MIN_SIZE = 60               # Minimum face bounding box dimension in pixels
MIN_BLUR = 20               # Laplacian variance threshold (accommodates WhatsApp/camera compression)
DETECTION_THRESHOLD = 0.90  # MTCNN detection confidence cutoff
MODEL_ID = "inceptionresnetv1-vggface2"
print("[VisuNav] Configuration loaded successfully.")

In [ ]:
# Initialize Face Detection & Embedding Models
detector = MTCNN(image_size=160, margin=20, keep_all=True, post_process=True, device=DEVICE)
encoder = InceptionResnetV1(pretrained="vggface2").eval().to(DEVICE)

def load_db():
    """Load the local face templates database."""
    if not DB_PATH.exists():
        return {"model": MODEL_ID, "contacts": {}}
    db = json.loads(DB_PATH.read_text(encoding="utf-8"))
    if db.get("model") != MODEL_ID:
        raise ValueError("Database embedding model mismatch: re-enroll contacts.")
    return db

def save_db(db):
    """Atomic write of database to prevent corruption."""
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    tmp = DB_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(db, indent=2), encoding="utf-8")
    tmp.replace(DB_PATH)

@torch.inference_mode()
def extract_faces(rgb, single_only=False):
    """Detect faces, validate quality, and extract 512-d normalized embeddings."""
    image = Image.fromarray(rgb)
    boxes, scores = detector.detect(image)
    if boxes is None or (single_only and len(boxes) != 1):
        return []
    
    # Explicit float conversion to ensure full NumPy 2.x compatibility
    boxes = np.asarray(boxes, dtype=np.float32)
    scores = np.asarray(scores, dtype=np.float32)
    h, w = rgb.shape[:2]
    accepted = []
    
    for box, score in zip(boxes, scores):
        if score is None or not np.isfinite(score) or score < DETECTION_THRESHOLD:
            continue
        x1, y1, x2, y2 = np.rint(box.astype(float)).astype(int)
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
        if min(x2 - x1, y2 - y1) < MIN_SIZE:
            continue
        crop = rgb[y1:y2, x1:x2]
        blur = cv2.Laplacian(cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var()
        if blur < MIN_BLUR:
            continue
        accepted.append([x1, y1, x2, y2])
        
    if not accepted:
        return []
        
    crops = detector.extract(image, np.asarray(accepted), save_path=None)
    if crops is None:
        return []
        
    embeddings = F.normalize(encoder(crops.to(DEVICE)), dim=1).cpu().numpy()
    return [{"box": box, "embedding": emb} for box, emb in zip(accepted, embeddings)]

def identify(embedding, db):
    """Match an embedding against registered templates using cosine similarity."""
    candidates = []
    for key, c in db["contacts"].items():
        if not c.get("consent"):
            continue
        template = np.asarray(c["template"], dtype=np.float32)
        if template.shape != (512,) or not np.isfinite(template).all():
            continue
        template = template / max(np.linalg.norm(template), 1e-12)
        similarity = float(np.dot(embedding, template))
        candidates.append((similarity, key))
        
    candidates.sort(reverse=True)
    if not candidates:
        return None, None
    best_sim, best_key = candidates[0]
    runner_up = candidates[1][0] if len(candidates) > 1 else -1.0
    
    # Identity confirmed only if threshold exceeded and separated from runner-up
    is_match = (best_sim >= THRESHOLD and (best_sim - runner_up) >= MARGIN)
    return (best_key if is_match else None), best_sim

## 1. Contact Enrollment
Enrolls a consenting contact by processing photos from their folder in `enrollment/<contact_name>/`.
Each photo is checked to contain exactly one clear, sharp face. The individual embeddings are averaged and L2-normalized into a master facial template.

In [ ]:
def enroll_contact(contact_id, name, relation, folder, consent=False):
    if not consent:
        raise ValueError("Explicit consent is required prior to biometric enrollment.")
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")
        
    paths = sorted(p for p in folder.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
    print(f"[Enrollment] Processing {len(paths)} images for {name} ({relation})...")
    
    embeddings = []
    for p in paths:
        try:
            with Image.open(p) as im:
                rgb = np.asarray(im.convert("RGB"))
            faces = extract_faces(rgb, single_only=True)
            if len(faces) == 1:
                embeddings.append(faces[0]["embedding"])
                print(f"  ✓ Accepted: {p.name}")
            else:
                print(f"  ✗ Rejected (expected 1 face, found {len(faces)} or blurred): {p.name}")
        except Exception as exc:
            print(f"  ✗ Error reading {p.name}: {exc}")
            
    if len(embeddings) < 3:
        raise ValueError(f"Insufficient samples ({len(embeddings)} accepted). Minimum 3 required.")
        
    # Create normalized master template
    master_template = np.mean(embeddings, axis=0)
    master_template /= max(np.linalg.norm(master_template), 1e-12)
    
    db = load_db()
    db["contacts"][contact_id] = {
        "name": name,
        "relation": relation,
        "consent": True,
        "samples": len(embeddings),
        "template": master_template.tolist()
    }
    save_db(db)
    print(f"\n Successfully Enrolled {name} ({relation}) with {len(embeddings)} valid sample templates!")

# Enroll Vinay Kasana from your dataset folder
enroll_contact(
    contact_id="vinay",
    name="Vinay Kasana",
    relation="friend",
    folder="enrollment/Vinay Kasana",
    consent=True
)

In [ ]:
# View All Enrolled Contacts in Database
db = load_db()
print(f"Database: {DB_PATH}")
print(f"Total Enrolled Contacts: {len(db.get('contacts', {}))}\n")
for cid, info in db.get("contacts", {}).items():
    print(f"• ID: {cid:<10} | Name: {info['name']:<15} | Relation: {info['relation']:<10} | Samples: {info['samples']}")

## 2. Model Inference & Visual Evaluation on Test Faces
Tests all images in `test_faces/` against the enrolled contacts database.
- Recognized contacts are highlighted with a **Green Box** and confidence score.
- Unrecognized faces (unknown individuals) are highlighted with an **Orange Box**.
- Results are plotted inline with `matplotlib` for immediate verification.

In [ ]:
# Run Inference on Test Faces & Display Visualizations
db = load_db()
test_files = sorted(list(TEST_FACES_DIR.glob("*.jpeg")) + list(TEST_FACES_DIR.glob("*.jpg")) + list(TEST_FACES_DIR.glob("*.png")))

if not test_files:
    print(f"No test images found in {TEST_FACES_DIR}")
else:
    print(f"Found {len(test_files)} test images for evaluation:\n" + "-"*60)
    
    for img_path in test_files:
        with Image.open(img_path) as im:
            rgb = np.asarray(im.convert("RGB"))
        annotated = rgb.copy()
        faces = extract_faces(rgb)
        
        print(f"Image: {img_path.name}")
        print(f"  Total faces detected: {len(faces)}")
        
        for i, face in enumerate(faces):
            x1, y1, x2, y2 = [int(v) for v in face["box"]]
            key, sim = identify(face["embedding"], db)
            
            if key:
                c = db["contacts"][key]
                label = f"{c['name']} ({c['relation']}) | {sim:.2f}"
                color = (0, 220, 0) # Green for recognized familiar contact
                print(f"  [Face {i+1}] RECOGNIZED: {c['name']} ({c['relation']}) - Similarity: {sim:.4f} (Threshold: {THRESHOLD})")
            else:
                sim_val = f"{sim:.2f}" if sim is not None else "N/A"
                label = f"Unknown | {sim_val}"
                color = (255, 120, 0) # Orange for unrecognized face
                print(f"  [Face {i+1}] UNRECOGNIZED: Similarity {sim_val} (Below threshold {THRESHOLD})")
                
            # Draw bounding box and label
            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 4)
            cv2.putText(annotated, label, (x1, max(30, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
            
        # Display inline plot
        plt.figure(figsize=(10, 7))
        plt.imshow(annotated)
        plt.axis("off")
        plt.title(f"VisuNav Recognition - {img_path.name}", fontsize=12, pad=10)
        plt.show()
        print("-"*60)

## 3. Assistive Speech Engine
Initializes an offline `pyttsx3` text-to-speech worker on a dedicated daemon thread.
Speech requests are queued non-blockingly so camera / navigation frame processing never lags.

In [ ]:
class Speaker:
    def __init__(self, enabled=True, speech_rate=165):
        self.enabled = enabled
        self.stop_event = threading.Event()
        self.q = queue.Queue(maxsize=1)
        self.thread = None
        if enabled:
            self.thread = threading.Thread(target=self._worker, args=(speech_rate,), daemon=True)
            self.thread.start()

    def _worker(self, rate):
        engine = None
        try:
            import pyttsx3
            engine = pyttsx3.init()
            engine.setProperty("rate", rate)
            while not self.stop_event.is_set():
                try:
                    msg = self.q.get(timeout=0.2)
                except queue.Empty:
                    continue
                if not self.stop_event.is_set():
                    engine.say(msg)
                    engine.runAndWait()
        except Exception as exc:
            print(f"[Audio Engine Unavailable]: {exc}")
        finally:
            if engine is not None:
                engine.stop()

    def say(self, msg):
        print(f"[Audio Announcement]: \"{msg}\"")
        if self.enabled:
            try:
                self.q.put_nowait(msg)
            except queue.Full:
                pass

    def close(self):
        self.stop_event.set()
        if self.thread:
            self.thread.join(timeout=2)

## 4. Real-Time Camera / Video Navigation
Runs real-time face recognition on live webcam (`SOURCE=0`) or recorded video.
- **Multi-frame Confirmation:** Requires consecutive matching frames before announcing identity.
- **Directional Audio:** Announces whether the person is on your left, right, or straight ahead.
- **Cooldown Control:** Prevents repetitive speech announcements for the same person within 15 seconds.

In [ ]:
def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / max(union, 1e-12)

def run_video(source=0, speech=True, confirmations=3, cooldown=15, width=960):
    db = load_db()
    if not db.get("contacts"):
        raise ValueError("No contacts found in database. Please run enrollment first.")
        
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(f"Unable to open video source: {source}")
        
    speaker = Speaker(enabled=speech)
    tracks = {}
    next_id = 1
    last_spoken = {}
    frame_id = 0
    
    print("[VisuNav] Video feed started. Press 'q' in the window to quit.")
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame_id += 1
            if frame.shape[1] > width:
                scale = width / frame.shape[1]
                frame = cv2.resize(frame, (width, round(frame.shape[0] * scale)))
                
            faces = extract_faces(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            tracks = {k: t for k, t in tracks.items() if frame_id - t["seen"] <= 2}
            unused = set(tracks.keys())
            
            for face in faces:
                key, score = identify(face["embedding"], db)
                box = face["box"]
                overlap, tid = max([(iou(box, tracks[k]["box"]), k) for k in unused], default=(0, None))
                
                if overlap < 0.3:
                    tid = next_id
                    next_id += 1
                    tracks[tid] = {"history": deque(maxlen=5)}
                else:
                    unused.remove(tid)
                    
                t = tracks[tid]
                if t.get("seen", frame_id - 1) != frame_id - 1:
                    t["history"].clear()
                t.update(box=box, seen=frame_id)
                t["history"].append(key)
                
                recent = list(t["history"])[-confirmations:]
                is_stable = (key is not None and len(recent) == confirmations and all(k == key for k in recent))
                x1, y1, x2, y2 = [int(v) for v in box]
                
                label = "Unrecognized face"
                color = (0, 165, 255) # Orange
                
                if is_stable:
                    c = db["contacts"][key]
                    label = f"{c['name']} ({c['relation']})"
                    color = (0, 220, 0) # Green
                    center_x = (x1 + x2) / (2 * frame.shape[1])
                    direction = "on your left" if center_x < 0.33 else "on your right" if center_x > 0.66 else "straight ahead"
                    now = time.monotonic()
                    if now - last_spoken.get(key, -float("inf")) >= cooldown:
                        speaker.say(f"{c['name']}, your {c['relation']}, is {direction}.")
                        last_spoken[key] = now
                elif key is not None:
                    label = "Verifying identity..."
                    
                if score is not None:
                    label += f" ({score:.2f})"
                    
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
                
            cv2.imshow("VisuNav Familiar Person Recognition - Press Q to Quit", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        speaker.close()
        print("[VisuNav] Video feed closed.")

# Set RUN_VIDEO = True when you are ready to test live camera / video
RUN_VIDEO = False
SOURCE = 0 # 0 for webcam, or specify path e.g. 'test_video.mp4'
if RUN_VIDEO:
    run_video(source=SOURCE, speech=True)
else:
    print("Set RUN_VIDEO = True and execute this cell to start live webcam recognition.")

## 5. Contact Management
Delete or update contact biometric records safely from `private_face_db/contacts.json`.

In [ ]:
DELETE_CONTACT_ID = ""  # Enter ID (e.g., 'vinay') to remove
if DELETE_CONTACT_ID:
    db = load_db()
    if DELETE_CONTACT_ID in db.get("contacts", {}):
        del db["contacts"][DELETE_CONTACT_ID]
        save_db(db)
        print(f"Contact '{DELETE_CONTACT_ID}' successfully removed from database.")
    else:
        print(f"Contact '{DELETE_CONTACT_ID}' not found in database.")
else:
    print("Specify DELETE_CONTACT_ID above if you wish to remove an enrolled contact.")